In [1]:
pip install opencv-python ultralytics pyyaml huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
"""
Environment and Import Verification Script
Author: Aleena K Xavier
"""

import sys

print("Python version:", sys.version)
print("-" * 40)

# 1. Test OpenCV
try:
    import cv2
    print(f"✓ OpenCV (cv2) imported successfully (version: {cv2.__version__})")
except ImportError as e:
    print(f"❌ FAILED to import OpenCV: {e}")

# 2. Test Ultralytics YOLO
try:
    from ultralytics import YOLO
    print("✓ Ultralytics YOLO imported successfully")
except ImportError as e:
    print(f"❌ FAILED to import Ultralytics: {e}")

# 3. Test PyYAML
try:
    import yaml
    print("✓ PyYAML imported successfully")
except ImportError as e:
    print(f"❌ FAILED to import PyYAML: {e}")

# 4. Test Hugging Face Hub
try:
    import huggingface_hub
    print("✓ Hugging Face Hub imported successfully")
except ImportError as e:
    print(f"❌ FAILED to import Hugging Face Hub: {e}")

# 5. Test Webcam Access
try:
    cap = cv2.VideoCapture(0)
    if cap.isOpened():
        print("✓ OpenCV webcam interface initialized successfully")
        cap.release()
    else:
        print("⚠️ Warning: OpenCV could not open webcam index 0 (this is normal if no physical webcam is connected right now).")
except Exception as e:
    print(f"⚠️ Webcam test warning: {e}")

print("-" * 40)
print("Import check complete!")

Python version: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
----------------------------------------
✓ OpenCV (cv2) imported successfully (version: 4.13.0)
✓ Ultralytics YOLO imported successfully
✓ PyYAML imported successfully
✓ Hugging Face Hub imported successfully
✓ OpenCV webcam interface initialized successfully
----------------------------------------
Import check complete!


In [4]:
import os
import csv
import threading
import time
import urllib.request
import cv2
from ultralytics import YOLO
from IPython.display import display, Image as IPyImage

# Windows sound fallback
try:
    import winsound
    HAS_WINSOUND = True
except ImportError:
    HAS_WINSOUND = False

# ==========================================
# 1. PROFILES & CONFIGURATION
# ==========================================
PROFILES = {
    "construction": {
        "description": "Standard construction site requiring hard hats, safety vests, and footwear",
        "required": {"helmet": 1, "safety-vest": 1, "shoes": 1}
    },
    "laboratory": {
        "description": "Chemical laboratory requiring eye protection, masks, and gloves",
        "required": {"glasses": 1, "face-mask": 1, "gloves": 1, "shoes": 1}
    },
    "factory": {
        "description": "Heavy manufacturing floor requiring full protection including earmuffs",
        "required": {"helmet": 1, "safety-vest": 1, "ear-mufs": 1, "shoes": 1}
    }
}

ACTIVE_PROFILE_NAME = "construction"
REQUIRED_ITEMS = PROFILES[ACTIVE_PROFILE_NAME]["required"]

DEFAULT_WEIGHTS = "sh17_yolo8s.pt"
SH17_WEIGHTS_URL = "https://github.com/ahmadmughees/SH17dataset/releases/download/v1/yolo8s.pt"
CONF_THRESH = 0.35

EXPECTED_REGION = {
    "helmet": (0.00, 0.35),
    "glasses": (0.00, 0.40),
    "face-mask": (0.00, 0.45),
    "face-guard": (0.00, 0.45),
    "ear-mufs": (0.00, 0.40),
    "shoes": (0.65, 1.00),
}

LOWER_BODY_ITEMS = {"shoes"}
CROP_MARGIN = 0.02
STATUS_COLORS = {
    "SAFE": (0, 255, 0),
    "UNSAFE": (0, 0, 255),
    "INCOMPLETE VIEW": (0, 165, 255),
}
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

print("Cell 1: Configuration loaded successfully.")

Cell 1: Configuration loaded successfully.


In [5]:
# ==========================================
# 2. MODEL DOWNLOADER & LOADER
# ==========================================
def ensure_weights(path):
    if os.path.exists(path):
        print(f"Weights already exist at: {path}")
        return

    print("Downloading SH17 weights (one-time download)...")
    tmp = path + ".part"
    try:
        req = urllib.request.Request(SH17_WEIGHTS_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(tmp, 'wb') as out_file:
            out_file.write(response.read())
        os.replace(tmp, path)
        print("Download complete.")
    except Exception as error:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise RuntimeError(
            f"Download failed: {error}\nDownload it in your browser from "
            f"{SH17_WEIGHTS_URL} and save it as {path}"
        )

ensure_weights(DEFAULT_WEIGHTS)
print(f"Loading detector: {DEFAULT_WEIGHTS}")
model = YOLO(DEFAULT_WEIGHTS)

# Safety check for label mismatches
known_classes = {str(n).lower() for n in model.names.values()}
for _prof_name, _prof in PROFILES.items():
    for _cls in _prof["required"]:
        if _cls not in known_classes:
            print(f"WARNING: profile '{_prof_name}' needs '{_cls}', which the model does not have")

Weights already exist at: sh17_yolo8s.pt
Loading detector: sh17_yolo8s.pt


c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\ultralytics\nn\tasks.py:732: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, ma

In [6]:
# ==========================================
# 3. DETECTION, SPATIAL & ALARM LOGIC
# ==========================================
def box_area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])

def in_expected_region(name, person_box, center_y):
    region = EXPECTED_REGION.get(name)
    if region is None:
        return True
    height = person_box[3] - person_box[1]
    if height <= 0:
        return False
    rel = (center_y - person_box[1]) / height
    return region[0] <= rel <= region[1]

_last_alarm = 0.0
def trigger_alarm():
    global _last_alarm
    now = time.time()
    if not HAS_WINSOUND or now - _last_alarm < 3.0:
        return
    _last_alarm = now
    threading.Thread(target=winsound.Beep, args=(2500, 300), daemon=True).start()

def detect(frame, model):
    h, w = frame.shape[:2]
    if w > 960:
        scale = 960 / w
        frame_resized = cv2.resize(frame, (960, int(h * scale)))
    else:
        scale = 1.0
        frame_resized = frame

    results = model(frame_resized, conf=CONF_THRESH, verbose=False)
    boxes = results[0].boxes

    persons, items = [], []
    for box in boxes:
        name = str(model.names[int(box.cls[0])]).lower()
        coords = list(map(int, box.xyxy[0]))
        if scale != 1.0:
            coords = [int(c / scale) for c in coords]

        conf = float(box.conf[0])
        if name == "person":
            persons.append({
                "box": coords,
                "id": len(persons) + 1,
                "confidence": conf,
                "ppe": {},
                "cropped_bottom": coords[3] >= h * (1 - CROP_MARGIN),
            })
        else:
            items.append((name, conf, coords))

    for name, _conf, coords in items:
        cx = (coords[0] + coords[2]) / 2
        cy = (coords[1] + coords[3]) / 2
        owners = [p for p in persons if p["box"][0] <= cx <= p["box"][2] and p["box"][1] <= cy <= p["box"][3]]
        if not owners:
            continue
        owner = min(owners, key=lambda p: box_area(p["box"]))
        if in_expected_region(name, owner["box"], cy):
            owner["ppe"][name] = owner["ppe"].get(name, 0) + 1

    return persons, items

def evaluate_person(person, required):
    missing, not_visible = [], []
    for cls, needed in required.items():
        if person["ppe"].get(cls, 0) >= needed:
            continue
        if cls in LOWER_BODY_ITEMS and person.get("cropped_bottom"):
            not_visible.append(cls.upper())
        else:
            missing.append(cls.upper())

    if missing:
        status = "UNSAFE"
    elif not_visible:
        status = "INCOMPLETE VIEW"
    else:
        status = "SAFE"
    return status, missing, not_visible

def process_frame(frame, model, required, alarm=True):
    persons, items = detect(frame, model)
    annotated = frame.copy()

    for name, conf, (x1, y1, x2, y2) in items:
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 165, 0), 1)
        cv2.putText(annotated, f"{name} {conf:.2f}", (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 165, 0), 1, cv2.LINE_AA)

    statuses = []
    for p in persons:
        status, missing, not_visible = evaluate_person(p, required)
        p["status"], p["missing"], p["not_visible"] = status, missing, not_visible
        statuses.append(status)

        color = STATUS_COLORS[status]
        x1, y1, x2, y2 = p["box"]
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        if status == "UNSAFE":
            text = f"WORKER #{p['id']} - UNSAFE (Missing: {', '.join(missing)})"
        elif status == "INCOMPLETE VIEW":
            text = f"WORKER #{p['id']} - OK so far (not visible: {', '.join(not_visible)})"
        else:
            text = f"WORKER #{p['id']} - SAFE"
        cv2.putText(annotated, text, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    if "UNSAFE" in statuses:
        overall = "UNSAFE"
        if alarm:
            trigger_alarm()
    elif "INCOMPLETE VIEW" in statuses:
        overall = "INCOMPLETE VIEW"
    else:
        overall = "SAFE"

    return annotated, overall, persons, items

def worker_details(p):
    parts = []
    if p["missing"]:
        parts.append(f"missing: {', '.join(p['missing'])}")
    if p["not_visible"]:
        parts.append(f"not visible: {', '.join(p['not_visible'])}")
    return f"  ({'; '.join(parts)})" if parts else ""

def print_worker_report(persons):
    for p in persons:
        print(f"Worker #{p['id']} -> {p['status']}{worker_details(p)}", flush=True)

print("Cell 3: Core logic functions loaded.")

Cell 3: Core logic functions loaded.


In [12]:
# ==========================================
# 4. BATCH TEST (Folder Processing)
# ==========================================
def run_batch(folder, profile_name, required):
    if not os.path.isdir(folder):
        print(f"Error: folder '{folder}' not found.")
        return

    files = sorted(f for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXTS))
    if not files:
        print(f"No images found in '{folder}'.")
        return

    out_dir = f"results_{profile_name}"
    os.makedirs(out_dir, exist_ok=True)
    rows, counts = [], {"SAFE": 0, "UNSAFE": 0, "INCOMPLETE VIEW": 0}

    for fname in files:
        img = cv2.imread(os.path.join(folder, fname))
        if img is None:
            continue

        annotated, overall, persons, items = process_frame(img, model, required, alarm=False)
        cv2.imwrite(os.path.join(out_dir, fname), annotated)

        if not persons:
            rows.append([fname, "", "NO WORKER DETECTED", "", "", "", "", ""])
            continue

        for p in persons:
            counts[p["status"]] += 1
            found = ", ".join(f"{k}:{v}" for k, v in sorted(p["ppe"].items()))
            rows.append([fname, p["id"], p["status"], found,
                         ", ".join(p["missing"]), ", ".join(p["not_visible"]), "", ""])

    csv_path = f"results_{profile_name}.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["image", "worker", "verdict", "detected_items", "missing",
                         "not_visible", "items actually worn (fill in)", "verdict correct y/n (fill in)"])
        writer.writerows(rows)

    print(f"\nBatch complete! Profile: {profile_name} | Images: {len(files)}")
    print(f"Counts -> SAFE: {counts['SAFE']}, UNSAFE: {counts['UNSAFE']}, INCOMPLETE: {counts['INCOMPLETE VIEW']}")
    print(f"Saved CSV table: {csv_path} and images folder: {out_dir}")

print("Cell 4: Batch test function loaded.")

Cell 4: Batch test function loaded.


In [28]:
# ==========================================
# 5. INTERACTIVE MENU LOOP
# ==========================================
print("=" * 50)
print("        INDUSTRIAL PPE COMPLIANCE SYSTEM")
print("=" * 50)

while True:
    print(f"\n[{ACTIVE_PROFILE_NAME.upper()}] needs: {', '.join(REQUIRED_ITEMS)}")
    print("1 = change scenario | 2 = test image | 3 = webcam | 4 = exit | 5 = test a folder")

    choice = input("Enter your choice (1-5): ").strip()

    if choice == "1":
        print("\nSelect a Safety Scenario:")
        print("  [c] Construction (Helmet, Safety Vest, Shoes)")
        print("  [l] Laboratory (Glasses, Mask, Gloves, Shoes)")
        print("  [f] Factory (Helmet, Safety Vest, Earmuffs, Shoes)")

        sub_choice = input("Enter profile letter (c/l/f): ").strip().lower()
        if sub_choice == "c":
            ACTIVE_PROFILE_NAME = "construction"
        elif sub_choice == "l":
            ACTIVE_PROFILE_NAME = "laboratory"
        elif sub_choice == "f":
            ACTIVE_PROFILE_NAME = "factory"
        else:
            print("Invalid choice. Keeping previous profile.")
            continue

        REQUIRED_ITEMS = PROFILES[ACTIVE_PROFILE_NAME]["required"]
        print(f"\nScenario changed to: {ACTIVE_PROFILE_NAME.upper()}\n")

    elif choice == "2":
        test_image_path = "test_constructionfolder/construction_worker2.jpg"

        if not os.path.exists(test_image_path):
            print(f"Error: File '{test_image_path}' not found.")
            continue

        img = cv2.imread(test_image_path)
        if img is None:
            print(f"Error: Could not read image '{test_image_path}'.")
            continue

        output_img, status, persons, items = process_frame(img, model, REQUIRED_ITEMS)
        result = status if persons else "NO WORKER DETECTED"

        print("\n" + "-" * 40, flush=True)
        print(f"EVALUATION REPORT [{ACTIVE_PROFILE_NAME.upper()}]", flush=True)
        print(f"RESULT: {result}", flush=True)
        print(f"Detected {len(persons)} worker(s) and {len(items)} PPE item(s)", flush=True)
        print("-" * 40, flush=True)
        print_worker_report(persons)
        print("-" * 40, flush=True)

        display(IPyImage(data=cv2.imencode(".jpg", output_img)[1].tobytes()))

    elif choice == "3":
        print(f"\nStarting live webcam feed [{ACTIVE_PROFILE_NAME.upper()} mode]...")
        print("Click the video window and press 'q' to return to the menu.")
        cap = cv2.VideoCapture(0)

        if not cap.isOpened():
            print("Error: Could not open webcam index 0.")
        else:
            last_summary, last_print = None, 0.0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    print("Failed to grab frame from camera.")
                    break

                output_frame, overall_status, persons, items = process_frame(frame, model, REQUIRED_ITEMS)
                cv2.imshow(f"Industrial PPE Compliance - Live Feed [{ACTIVE_PROFILE_NAME.upper()}]", output_frame)

                summary = tuple((p["id"], p["status"]) for p in persons)
                now = time.time()
                if summary != last_summary and now - last_print >= 1.0:
                    last_summary, last_print = summary, now
                    if persons:
                        print_worker_report(persons)
                    else:
                        print("No worker in view", flush=True)

                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

            cap.release()
            cv2.destroyAllWindows()
            print("Live webcam feed closed. Returning to menu...")

    elif choice == "4":
        print("Exiting Industrial PPE Compliance System.")
        break

    elif choice == "5":
        folder = input("Folder with test images [test_images]: ").strip().strip('"') or "test_images"
        run_batch(folder, ACTIVE_PROFILE_NAME, REQUIRED_ITEMS)

    else:
        print("Invalid choice. Please enter a number between 1 and 5.")

        INDUSTRIAL PPE COMPLIANCE SYSTEM

[FACTORY] needs: helmet, safety-vest, ear-mufs, shoes
1 = change scenario | 2 = test image | 3 = webcam | 4 = exit | 5 = test a folder

Select a Safety Scenario:
  [c] Construction (Helmet, Safety Vest, Shoes)
  [l] Laboratory (Glasses, Mask, Gloves, Shoes)
  [f] Factory (Helmet, Safety Vest, Earmuffs, Shoes)

Scenario changed to: LABORATORY


[LABORATORY] needs: glasses, face-mask, gloves, shoes
1 = change scenario | 2 = test image | 3 = webcam | 4 = exit | 5 = test a folder

Starting live webcam feed [LABORATORY mode]...
Click the video window and press 'q' to return to the menu.
Worker #1 -> UNSAFE  (missing: FACE-MASK, GLOVES; not visible: SHOES)
Worker #1 -> UNSAFE  (missing: FACE-MASK, GLOVES; not visible: SHOES)
Worker #2 -> UNSAFE  (missing: GLASSES, FACE-MASK, GLOVES; not visible: SHOES)
Worker #1 -> UNSAFE  (missing: FACE-MASK, GLOVES; not visible: SHOES)
Worker #1 -> UNSAFE  (missing: GLASSES, FACE-MASK, GLOVES; not visible: SHOES)
W

In [24]:
import cv2
img = cv2.imread("test_constructionfolder/construction_worker2.jpg")
persons, items = detect(img, model)

print("DETECTED (confidence >= 0.35):")
for name, conf, box in sorted(items):
    print(f"  {name:14s} {conf:.2f}")
for p in persons:
    print("Worker", p["id"], "assigned:", p["ppe"])

# Lower threshold, to see weak detections the normal setting hides
r = model(img, conf=0.10, verbose=False)[0]
print("\nWEAK DETECTIONS (confidence >= 0.10):")
for b in r.boxes:
    print(f"  {model.names[int(b.cls[0])]:14s} {float(b.conf[0]):.2f}")

DETECTED (confidence >= 0.35):
  ear            0.57
  face           0.91
  glasses        0.40
  gloves         0.82
  gloves         0.86
  head           0.89
  helmet         0.89
  safety-vest    0.84
  safety-vest    0.85
  shoes          0.47
  shoes          0.55
Worker 1 assigned: {'face': 1, 'helmet': 1, 'head': 1, 'gloves': 2, 'safety-vest': 2, 'ear': 1, 'shoes': 2, 'glasses': 1}

WEAK DETECTIONS (confidence >= 0.10):
  face           0.91
  helmet         0.90
  head           0.89
  gloves         0.88
  safety-vest    0.87
  person         0.87
  gloves         0.81
  safety-vest    0.81
  ear            0.63
  shoes          0.60
  glasses        0.35
  shoes          0.29
  ear            0.26
  ear            0.26
  person         0.25
  person         0.12
  person         0.11
  glasses        0.11
